# Análise Exploratória de Dados (EDA) & Modelagem Estatística — Mercado de Ações B3 (2021-2023)
**Projeto:** Performance B3 (Linha 2) — Business Intelligence & Visualização de Dados  
**Autoria Técnica:** Andressa Alves de Souza  
**Orientação Pedagógica:** Prof. Sávio Ricardo Garcia  
**Objetivo:** Prototipar as métricas quantitativas, validar hipóteses financeiras, calcular os 4 KPIs executivos da carteira e responder visual e estatisticamente às 5 perguntas de negócio da B3 com Plotly Express.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go

# Configurações globais de exibição
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)

print(f"Ambiente ativo: Python {sys.version.split()[0]} | Pandas {pd.__version__} | Plotly {plotly.__version__}")

Ambiente ativo: Python 3.12.3 | Pandas 2.3.3 | Plotly 5.24.1


## 1. Ingestão Resiliente e Auditoria de Qualidade dos Dados
Carregamento da base processada da Camada Ouro (`data/processed/b3_stocks_processed.csv`) utilizando caminhos estritamente relativos com `pathlib.Path`, assegurando a execução no Streamlit Cloud sem caminhos absolutos locais.

In [2]:
# Mapeamento dinâmico e resiliente do diretório raiz do projeto
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = BASE_DIR / "data" / "processed" / "b3_stocks_processed.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Ficheiro de dados não encontrado no caminho relativo: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

# Padronização de datas e ordenação cronológica estrita
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(by=["ticker", "date"]).reset_index(drop=True)

# Auditoria formal de integridade matemática e dimensional
print("=" * 65)
print(f"Dimensão da Base (Linhas, Colunas): {df.shape}")
print(f"Ativos Analisados ({df['ticker'].nunique()} papéis): {sorted(df['ticker'].unique())}")
print(f"Janela Histórica: {df['date'].min().strftime('%d/%m/%Y')} a {df['date'].max().strftime('%d/%m/%Y')}")
print(f"Total de Pregões por Ativo: {len(df) // df['ticker'].nunique()} sessões")
print("=" * 65)

# Asserções de integridade técnica
assert df.isna().sum().sum() == 0, "Falha: Existem valores nulos na base saneada."
assert (df["drawdown"] <= 1e-6).all(), "Falha: Existem registos de Drawdown com valor positivo."
assert (df.groupby("ticker").first()["base_100"] == 100.0).all(), "Falha: Base 100 não se inicia ancorada em 100.0."

print("Auditoria concluída com sucesso: 100% de integridade e conformidade.")
df.info()

Dimensão da Base (Linhas, Colunas): (8940, 14)
Ativos Analisados (12 papéis): ['ABEV3.SA', 'BBDC4.SA', 'EGIE3.SA', 'EMBJ3.SA', 'GGBR4.SA', 'ITUB4.SA', 'KLBN11.SA', 'PETR4.SA', 'PRIO3.SA', 'RENT3.SA', 'VALE3.SA', 'WEGE3.SA']
Janela Histórica: 04/01/2021 a 28/12/2023
Total de Pregões por Ativo: 745 sessões
Auditoria concluída com sucesso: 100% de integridade e conformidade.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8940 entries, 0 to 8939
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        8940 non-null   datetime64[ns]
 1   ticker      8940 non-null   object        
 2   setor       8940 non-null   object        
 3   open        8940 non-null   float64       
 4   high        8940 non-null   float64       
 5   low         8940 non-null   float64       
 6   adj_close   8940 non-null   float64       
 7   volume      8940 non-null   float64       
 8   ret_diario  8940 non-null   float

## 2. Auditoria Estatística Descritiva dos Retornos Diários
Investigação dos momentos estatísticos da série: Média Diária, Desvio Padrão ($\sigma$), Volatilidade Anualizada ($\sigma \times \sqrt{252}$), Assimetria (*Skewness*), Curtose (*Kurtosis*), Retorno Acumulado, Índice Sharpe Empírico e Rebaixamento Máximo (*Max Drawdown*).

In [3]:
stats_list = []

for ticker, group in df.groupby("ticker"):
    # Exclusão do primeiro pregão para cálculo correto dos retornos diários
    ret = group["ret_diario"].iloc[1:]
    setor = group["setor"].iloc[0]
    
    media_diaria = ret.mean()
    std_diario = ret.std()
    vol_anual = std_diario * np.sqrt(252)
    skew = ret.skew()
    kurt = ret.kurtosis()
    
    # Retorno acumulado ancorado na Base 100 e Max Drawdown da série
    base_100_final = group["base_100"].iloc[-1]
    ret_acumulado = (base_100_final - 100.0) / 100.0
    max_dd = group["drawdown"].min()
    
    # Índice Sharpe Empírico (anualizado)
    sharpe = ret_acumulado / vol_anual if vol_anual > 0 else 0.0
    
    stats_list.append({
        "Ticker": ticker,
        "Setor": setor,
        "Retorno Acumulado (%)": ret_acumulado * 100,
        "Média Diária (%)": media_diaria * 100,
        "Desvio Padrão Diário (%)": std_diario * 100,
        "Volatilidade Anualizada (%)": vol_anual * 100,
        "Índice Sharpe": sharpe,
        "Max Drawdown (%)": max_dd * 100,
        "Assimetria (Skewness)": skew,
        "Curtose (Kurtosis)": kurt
    })

df_stats = pd.DataFrame(stats_list).sort_values(by="Retorno Acumulado (%)", ascending=False).reset_index(drop=True)
df_stats.round(2)

,Ticker,Setor,Retorno Acumulado (%),Média Diária (%),Desvio Padrão Diário (%),Volatilidade Anualizada (%),Índice Sharpe,Max Drawdown (%),Assimetria (Skewness),Curtose (Kurtosis)
0,PETR4.SA,"Petróleo, Gás & Energia",228.69,0.19,2.48,39.42,5.80,-39.10,-0.87,7.63
1,PRIO3.SA,"Petróleo, Gás & Energia",208.32,0.19,2.95,46.78,4.45,-34.32,0.30,2.19
2,EMBJ3.SA,Bens de Capital & Indústria,167.50,0.18,2.95,46.82,3.58,-58.27,0.59,4.24
3,GGBR4.SA,Materiais Básicos & Siderurgia/Papel,30.79,0.06,2.24,35.56,0.87,-33.93,0.10,0.79
4,EGIE3.SA,"Petróleo, Gás & Energia",25.35,0.04,1.18,18.76,1.35,-17.65,0.03,1.61
5,ITUB4.SA,"Financeiro, Consumo & Locação",24.02,0.05,1.82,28.83,0.83,-35.87,-1.11,13.49
6,VALE3.SA,Materiais Básicos & Siderurgia/Papel,18.11,0.05,2.14,33.90,0.53,-41.34,0.40,1.51
7,WEGE3.SA,Bens de Capital & Indústria,3.51,0.03,2.07,32.83,0.11,-49.51,0.39,2.30
8,ABEV3.SA,"Financeiro, Consumo & Locação",2.57,0.02,1.60,25.46,0.10,-31.42,0.53,3.46
9,RENT3.SA,"Financeiro, Consumo & Locação",-0.25,0.03,2.32,36.89,-0.01,-38.40,0.18,1.29


## 3. Síntese dos 4 KPIs Estratégicos Oficiais da Carteira (Critério 10)
Consolidação dos quatro indicadores executivos para o cabeçalho do painel de BI (Padrão Pirâmide Invertida com cartões `st.metric`):
1. **Retorno Médio da Carteira (%)**
2. **Ativo Destaque da Carteira (Maior rentabilidade acumulada na Base 100)**
3. **Ativo de Menor Performance (Menor rentabilidade acumulada na Base 100)**
4. **Volatilidade Anualizada Média da Carteira (% a.a.)**

In [4]:
kpi_retorno_medio = df_stats["Retorno Acumulado (%)"].mean()
kpi_destaque = df_stats.loc[df_stats["Retorno Acumulado (%)"].idxmax()]
kpi_menor = df_stats.loc[df_stats["Retorno Acumulado (%)"].idxmin()]
kpi_vol_media = df_stats["Volatilidade Anualizada (%)"].mean()

print("=" * 65)
print("             SÍNTESE DOS 4 KPIS OFICIAIS DA CARTEIRA")
print("=" * 65)
print(f"1. Retorno Médio da Carteira:       {kpi_retorno_medio:+.2f}%")
print(f"2. Ativo Destaque (Top 1):          {kpi_destaque['Ticker']} ({kpi_destaque['Retorno Acumulado (%)']:+.2f}%) [{kpi_destaque['Setor']}]")
print(f"3. Ativo de Menor Performance:      {kpi_menor['Ticker']} ({kpi_menor['Retorno Acumulado (%)']:+.2f}%) [{kpi_menor['Setor']}]")
print(f"4. Volatilidade Anualizada Média:   {kpi_vol_media:.2f}% a.a.")
print("=" * 65)

             SÍNTESE DOS 4 KPIS OFICIAIS DA CARTEIRA
1. Retorno Médio da Carteira:       +58.41%
2. Ativo Destaque (Top 1):          PETR4.SA (+228.69%) [Petróleo, Gás & Energia]
3. Ativo de Menor Performance:      BBDC4.SA (-6.93%) [Financeiro, Consumo & Locação]
4. Volatilidade Anualizada Média:   33.61% a.a.


## 4. Resposta e Prototipagem das 5 Perguntas de Negócio Oficiais 

### Pergunta de Negócio 1:
> *Se um investidor aplicasse capital em cada ativo no início de 2021, qual seria a rentabilidade comparativa normalizada (Base 100) acumulada até o final de 2023?*

**Justificativa Analítica:**  
A Base 100 iguala o ponto de entrada de todos os ativos independentemente da sua cotação nominal em reais. A linha tracejada vermelha no nível 100.0 define a fronteira de valorização e preservação de capital.

In [14]:
# Gráfico 1: Versão Executiva
destaque_top = df_stats.loc[df_stats["Retorno Acumulado (%)"].idxmax()]["Ticker"]
destaque_bot = df_stats.loc[df_stats["Retorno Acumulado (%)"].idxmin()]["Ticker"]
destaque_def = "EGIE3.SA"

fig_base100 = go.Figure()

# 1. Ativos intermediários em cinza neutro
for t in sorted(df["ticker"].unique()):
    if t not in [destaque_top, destaque_bot, destaque_def]:
        df_sub = df[df["ticker"] == t]
        ret_val = (df_sub["base_100"].iloc[-1] - 100.0)
        fig_base100.add_trace(go.Scatter(
            x=df_sub["date"],
            y=df_sub["base_100"],
            mode="lines",
            name=f"{t} ({ret_val:+.1f}%)",
            line=dict(color="rgba(160, 174, 192, 0.40)", width=1.3),
            hovertemplate="<b>%{x|%d/%m/%Y}</b><br>Ativo: " + str(t) + "<br>Base 100: <b>%{y:.2f}</b><extra></extra>"
        ))

# 2. Ativos estratégicos de destaque
# Top 1 - Maior Rentabilidade
df_top = df[df["ticker"] == destaque_top]
ret_top = df_top["base_100"].iloc[-1] - 100.0
fig_base100.add_trace(go.Scatter(
    x=df_top["date"],
    y=df_top["base_100"],
    mode="lines",
    name=f"★ Destaque: {destaque_top} ({ret_top:+.1f}%)",
    line=dict(color="#00875A", width=3.0),
    hovertemplate="<b>%{x|%d/%m/%Y}</b><br>Destaque: <b>" + str(destaque_top) + "</b><br>Base 100: <b>%{y:.2f}</b><extra></extra>"
))

# Ativo Defensivo / Mais Resiliente
df_def = df[df["ticker"] == destaque_def]
ret_def = df_def["base_100"].iloc[-1] - 100.0
fig_base100.add_trace(go.Scatter(
    x=df_def["date"],
    y=df_def["base_100"],
    mode="lines",
    name=f"◆ Resiliente: {destaque_def} ({ret_def:+.1f}%)",
    line=dict(color="#0052CC", width=2.4),
    hovertemplate="<b>%{x|%d/%m/%Y}</b><br>Resiliente: <b>" + str(destaque_def) + "</b><br>Base 100: <b>%{y:.2f}</b><extra></extra>"
))

# Menor Desempenho
df_bot = df[df["ticker"] == destaque_bot]
ret_bot = df_bot["base_100"].iloc[-1] - 100.0
fig_base100.add_trace(go.Scatter(
    x=df_bot["date"],
    y=df_bot["base_100"],
    mode="lines",
    name=f"▼ Menor Retorno: {destaque_bot} ({ret_bot:+.1f}%)",
    line=dict(color="#DE350B", width=2.6),
    hovertemplate="<b>%{x|%d/%m/%Y}</b><br>Menor Retorno: <b>" + str(destaque_bot) + "</b><br>Base 100: <b>%{y:.2f}</b><extra></extra>"
))

# 3. Linha de Suporte de Capital (100.0)
fig_base100.add_hline(
    y=100.0,
    line_dash="dash",
    line_color="#475569",
    line_width=1.5,
    annotation_text="Preservação de Capital (Base = 100.0)",
    annotation_position="top right",
    annotation_font=dict(size=10, color="#475569")
)

# 4. Layout panorâmico
fig_base100.update_layout(
    title=dict(
        text="<b>Rentabilidade Comparativa Normalizada na Base 100 (2021-2023)</b><br>"
             "<span style='font-size: 12px; color: #64748B;'>Destaque estratégico de dispersão entre líderes de ciclo, ativos defensivos e sob pressão macroeconômica</span>",
        font=dict(size=15, color="#0F172A"),
        x=0.01
    ),
    xaxis=dict(
        title="Data do Pregão",
        showgrid=True,
        gridcolor="rgba(226, 232, 240, 0.7)",
        tickformat="%b/%Y"
    ),
    yaxis=dict(
        title="Performance Normalizada (Base 100)",
        showgrid=True,
        gridcolor="rgba(226, 232, 240, 0.7)"
    ),
    template="plotly_white",
    height=580,
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.18,
        xanchor="center",
        x=0.5,
        font=dict(size=9.5),
        bgcolor="rgba(255, 255, 255, 0.9)",
        bordercolor="rgba(226, 232, 240, 0.8)",
        borderwidth=1
    ),
    margin=dict(r=30, t=75, b=90, l=55)
)

fig_base100.show()

### Pergunta de Negócio 2:
> *Quais ativos entregaram a melhor relação de Sharpe empírica (maior retorno acumulado com menor volatilidade histórica anualizada)?*

**Justificativa Analítica:**  
Dispersão de Risco vs. Retorno cruzando Volatilidade Anualizada no eixo X com Retorno Acumulado no eixo Y. As bolhas são dimensionadas pelo Volume Médio Diário e segmentadas pela cor do setor económico. O quadrante superior esquerdo define a máxima eficiência empírica.

In [29]:
# Gráfico 2: Mapa de Eficiência Risco x Retorno (Construção Robusta via Graph Objects - Zero Sobreposição)
# 1. Base consolidada e padronização setorial oficial
MAPA_SETORES_OFICIAL = {
    "PETR4.SA": "Petróleo, Gás & Energia",
    "PRIO3.SA": "Petróleo, Gás & Energia",
    "EGIE3.SA": "Petróleo, Gás & Energia",
    "ITUB4.SA": "Financeiro, Consumo & Locação",
    "BBDC4.SA": "Financeiro, Consumo & Locação",
    "ABEV3.SA": "Financeiro, Consumo & Locação",
    "RENT3.SA": "Financeiro, Consumo & Locação",
    "WEGE3.SA": "Bens de Capital & Indústria",
    "EMBJ3.SA": "Bens de Capital & Indústria",
    "VALE3.SA": "Materiais Básicos & Siderurgia/Papel",
    "GGBR4.SA": "Materiais Básicos & Siderurgia/Papel",
    "KLBN11.SA": "Materiais Básicos & Siderurgia/Papel"
}

df_vol = df.groupby("ticker", as_index=False)["volume"].mean()
df_vol.columns = ["Ticker", "Volume_Medio"]

df_scatter = pd.merge(df_stats[["Ticker", "Retorno Acumulado (%)", "Volatilidade Anualizada (%)"]], df_vol, on="Ticker")
df_scatter["Setor"] = df_scatter["Ticker"].map(MAPA_SETORES_OFICIAL)

# 2. Paleta cromática oficial e fator de escala de bolhas
paleta_oficial = {
    "Petróleo, Gás & Energia": "#059669",             # Verde Esmeralda
    "Financeiro, Consumo & Locação": "#2563EB",       # Azul Royal
    "Bens de Capital & Indústria": "#EA580C",         # Laranja Terracota
    "Materiais Básicos & Siderurgia/Papel": "#7E22CE" # Roxo Uva
}

vol_max = df_scatter["Volume_Medio"].max()

fig_sharpe = go.Figure()

# 3. Adiciona as bolhas agrupadas por setor (garantindo legenda perfeita e cores sem mistura)
for setor_nome, cor in paleta_oficial.items():
    df_setor = df_scatter[df_scatter["Setor"] == setor_nome]
    if df_setor.empty:
        continue
    
    # Normalização de tamanho entre 14px e 40px proporcional ao volume
    tamanhos = 14 + (df_setor["Volume_Medio"] / vol_max) * 26
    
    fig_sharpe.add_trace(go.Scatter(
        x=df_setor["Volatilidade Anualizada (%)"],
        y=df_setor["Retorno Acumulado (%)"],
        mode="markers",
        name=setor_nome,
        marker=dict(
            size=tamanhos,
            color=cor,
            opacity=0.90,
            line=dict(width=1.2, color="#0F172A")
        ),
        text=df_setor["Ticker"],
        customdata=df_setor[["Volume_Medio", "Setor"]].values,
        hovertemplate="<b>%{text}</b><br>" +
                      "Setor: <b>%{customdata[1]}</b><br>" +
                      "Retorno Acumulado: <b>%{y:+.2f}%</b><br>" +
                      "Volatilidade Anualizada: <b>%{x:.2f}% a.a.</b><br>" +
                      "Volume Médio Diário: <b>R$ %{customdata[0]:,.0f}</b><extra></extra>"
    ))

# 4. Linha horizontal de preservação de capital / retorno neutro
fig_sharpe.add_hline(y=0.0, line_dash="dash", line_color="#94A3B8", line_width=1.2)

# 5. Rótulos individuais com deslocamentos precisos (ZERO sobreposição mecânica)
# PRIO3.SA puxado 18px para CIMA da sua bolha
# EMBJ3.SA puxado 20px para BAIXO da sua bolha
posicoes_texto = {
    "PETR4.SA": dict(yshift=20, xshift=0),
    "PRIO3.SA": dict(yshift=18, xshift=0),       # ACIMA da bolha verde
    "EMBJ3.SA": dict(yshift=-20, xshift=0),      # ABAIXO da bolha laranja
    "EGIE3.SA": dict(yshift=14, xshift=0),
    "ITUB4.SA": dict(yshift=18, xshift=0),
    "VALE3.SA": dict(yshift=16, xshift=-12),
    "GGBR4.SA": dict(yshift=16, xshift=12),
    "RENT3.SA": dict(yshift=14, xshift=14),
    "ABEV3.SA": dict(yshift=-16, xshift=-10),    # Abaixo da bolha
    "KLBN11.SA": dict(yshift=16, xshift=12),     # Acima da bolha
    "BBDC4.SA": dict(yshift=-20, xshift=-8),     # Abaixo da bolha
    "WEGE3.SA": dict(yshift=16, xshift=8)        # Acima da bolha
}

for _, row in df_scatter.iterrows():
    t = row["Ticker"]
    pos = posicoes_texto.get(t, dict(yshift=14, xshift=0))
    fig_sharpe.add_annotation(
        x=row["Volatilidade Anualizada (%)"],
        y=row["Retorno Acumulado (%)"],
        text=f"<b>{t}</b>",
        showarrow=False,
        yshift=pos["yshift"],
        xshift=pos["xshift"],
        font=dict(size=9.5, family="Segoe UI, Arial, sans-serif", color="#0F172A")
    )

# 6. Layout corporativo em tela única sem rolagem vertical
fig_sharpe.update_layout(
    title=dict(
        text="<b>Mapa de Eficiência: Risco (Volatilidade Anualizada) vs. Retorno Acumulado (2021-2023)</b><br>"
             "<span style='font-size: 11px; color: #64748B;'>Tamanho da bolha proporcional ao Volume Médio Diário (R$) | Quadrante superior esquerdo = Maior Eficiência</span>",
        font=dict(size=14, color="#0F172A"),
        x=0.01
    ),
    xaxis=dict(
        title="Volatilidade Anualizada Histórica (% a.a.)",
        showgrid=True,
        gridcolor="rgba(226, 232, 240, 0.7)",
        ticksuffix="%",
        range=[15, 55]
    ),
    yaxis=dict(
        title="Retorno Total Acumulado (%)",
        showgrid=True,
        gridcolor="rgba(226, 232, 240, 0.7)",
        ticksuffix="%",
        range=[-35, 275]
    ),
    hoverlabel=dict(
        font_color="#FFFFFF",
        bordercolor="#FFFFFF",
        font_size=11,
        font_family="Segoe UI, Arial, sans-serif"
    ),
    legend=dict(
        title_text="<b>Setor Econômico:</b>",
        orientation="h",
        yanchor="top",
        y=-0.22,
        xanchor="center",
        x=0.5,
        font=dict(size=9.5, color="#0F172A"),
        bgcolor="rgba(255, 255, 255, 0.9)",
        bordercolor="rgba(226, 232, 240, 0.8)",
        borderwidth=1
    ),
    template="plotly_white",
    height=490,
    margin=dict(r=30, t=70, b=80, l=60)
)

fig_sharpe.show()

### Pergunta de Negócio 3:
> *Como se comportam estatisticamente os retornos diários dos ativos (dispersão, assimetria e caudas de risco de mercado)?*

**Justificativa Analítica:**  
Histograma de distribuição de frequência com boxplot marginal superior, permitindo inspecionar visualmente a presença de assimetria e caudas grossas (*fat tails*) nos retornos diários dos papéis negociados na B3.

In [40]:
# Gráfico 3: Distribuição e Caudas de Risco dos Retornos Diários
from plotly.subplots import make_subplots
import plotly.graph_objects as go

df_ret = df[df["ret_diario"] != 0.0].copy()

media_ret = df_ret["ret_diario"].mean()
std_ret = df_ret["ret_diario"].std()
lim_inf = media_ret - 2 * std_ret
lim_sup = media_ret + 2 * std_ret

# 1. Dois painéis compactos com controle estrito de altura
fig_dist = make_subplots(
    rows=2, cols=1,
    row_heights=[0.22, 0.78],
    shared_xaxes=True,
    vertical_spacing=0.02
)

# 2. Boxplot superior limpo (sem hover invasivo)
fig_dist.add_trace(
    go.Box(
        x=df_ret["ret_diario"],
        name="",
        marker=dict(color="#2563EB", size=3.5, opacity=0.7),
        line=dict(color="#1E3A8A", width=1.2),
        fillcolor="rgba(37, 99, 235, 0.35)",
        orientation="h",
        showlegend=False,
        hoverinfo="none"
    ),
    row=1, col=1
)

# 3. Histograma inferior
fig_dist.add_trace(
    go.Histogram(
        x=df_ret["ret_diario"],
        nbinsx=100,
        marker=dict(
            color="#2563EB",
            opacity=0.82,
            line=dict(width=0.8, color="#1E3A8A")
        ),
        showlegend=False,
        hovertemplate="Intervalo: <b>%{x:.1%}</b><br>Pregões Registrados: <b>%{y} sessões</b><extra></extra>"
    ),
    row=2, col=1
)

# 4. Linhas verticais de dispersão
fig_dist.add_vline(x=0.0, line_dash="dash", line_color="#0F172A", line_width=1.4)
fig_dist.add_vline(x=lim_inf, line_dash="dot", line_color="#DC2626", line_width=1.3)
fig_dist.add_vline(x=lim_sup, line_dash="dot", line_color="#16A34A", line_width=1.3)

# 5. Anotações posicionadas na linha divisória limpa (ZERO colisão com subtítulo)
fig_dist.add_annotation(
    x=0.0, y=0.96, xref="x", yref="paper", yanchor="bottom",
    text="<b>Média (~0.0%)</b>",
    showarrow=False,
    font=dict(size=9.5, color="#0F172A"),
    bgcolor="rgba(255,255,255,0.85)"
)
fig_dist.add_annotation(
    x=lim_inf, y=0.96, xref="x", yref="paper", yanchor="bottom",
    text=f"<b>-2σ ({lim_inf:.1%})</b>",
    showarrow=False,
    font=dict(size=9.5, color="#DC2626"),
    bgcolor="rgba(255,255,255,0.85)"
)
fig_dist.add_annotation(
    x=lim_sup, y=0.96, xref="x", yref="paper", yanchor="bottom",
    text=f"<b>+2σ ({lim_sup:.1%})</b>",
    showarrow=False,
    font=dict(size=9.5, color="#16A34A"),
    bgcolor="rgba(255,255,255,0.85)"
)

# 6. Layout compacto para visualização imediata sem rolagem no VS Code
fig_dist.update_layout(
    title=dict(
        text="<b>Distribuição Empírica e Caudas de Risco dos Retornos Diários (Consolidado B3)</b><br>"
             "<span style='font-size: 11px; color: #64748B;'>Inspeção de dispersão, leptocurtose e eventos extremos de cauda (outliers além de ±2σ)</span>",
        font=dict(size=13.5, color="#0F172A"),
        x=0.01
    ),
    template="plotly_white",
    height=440,
    hoverlabel=dict(
        bgcolor="#1E293B",
        font_color="#FFFFFF",
        bordercolor="#FFFFFF",
        font_size=11,
        font_family="Segoe UI, Arial, sans-serif"
    ),
    margin=dict(r=30, t=65, b=55, l=60)
)

fig_dist.update_xaxes(showticklabels=False, row=1, col=1)
fig_dist.update_xaxes(
    title_text="Retorno Percentual Diário",
    tickformat=".1%",
    showgrid=True,
    gridcolor="rgba(226, 232, 240, 0.7)",
    range=[-0.12, 0.12],
    row=2, col=1
)
fig_dist.update_yaxes(showticklabels=False, showgrid=False, row=1, col=1)
fig_dist.update_yaxes(
    title_text="Frequência (Nº de Sessões)",
    showgrid=True,
    gridcolor="rgba(226, 232, 240, 0.7)",
    row=2, col=1
)

fig_dist.show()

### Pergunta de Negócio 4:
> *Qual foi o Drawdown Máximo percentual enfrentado por cada papel a partir de seu pico histórico e qual ativo se mostrou mais resiliente?*

**Justificativa Analítica:**  
Visualização contínua em gráfico de área do rebaixamento de cotação em relação ao pico histórico, acompanhado pelo ranking objetivo dos ativos de menor perda acumulada.

In [46]:
# Gráfico 4: Matriz de Drawdown Individual
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 1. Mapeamento e paleta oficial por departamento/setor
MAPA_SETORES_OFICIAL = {
    "PETR4.SA": "Petróleo, Gás & Energia",
    "PRIO3.SA": "Petróleo, Gás & Energia",
    "EGIE3.SA": "Petróleo, Gás & Energia",
    "ITUB4.SA": "Financeiro, Consumo & Locação",
    "BBDC4.SA": "Financeiro, Consumo & Locação",
    "ABEV3.SA": "Financeiro, Consumo & Locação",
    "RENT3.SA": "Financeiro, Consumo & Locação",
    "WEGE3.SA": "Bens de Capital & Indústria",
    "EMBJ3.SA": "Bens de Capital & Indústria",
    "VALE3.SA": "Materiais Básicos & Siderurgia/Papel",
    "GGBR4.SA": "Materiais Básicos & Siderurgia/Papel",
    "KLBN11.SA": "Materiais Básicos & Siderurgia/Papel"
}

paleta_oficial = {
    "Petróleo, Gás & Energia": "#059669",             # Verde Esmeralda
    "Financeiro, Consumo & Locação": "#2563EB",       # Azul Royal
    "Bens de Capital & Indústria": "#EA580C",         # Laranja Terracota
    "Materiais Básicos & Siderurgia/Papel": "#7E22CE" # Roxo Uva
}

paleta_rgb_fill = {
    "Petróleo, Gás & Energia": "rgba(5, 150, 105, 0.20)",
    "Financeiro, Consumo & Locação": "rgba(37, 99, 235, 0.20)",
    "Bens de Capital & Indústria": "rgba(234, 88, 12, 0.20)",
    "Materiais Básicos & Siderurgia/Papel": "rgba(126, 34, 206, 0.20)"
}

ativos_ordenados = sorted(list(MAPA_SETORES_OFICIAL.keys()), key=lambda x: (MAPA_SETORES_OFICIAL[x], x))
max_dd_dict = df_stats.set_index("Ticker")["Max Drawdown (%)"].to_dict()

titulos_sub = [f"<b>{t}</b> (Max: {max_dd_dict.get(t, 0.0):.1f}%)" for t in ativos_ordenados]

# 2. Grade 4x3 com espaçamento vertical ajustado
fig_dd = make_subplots(
    rows=4, cols=3,
    subplot_titles=titulos_sub,
    vertical_spacing=0.08,
    horizontal_spacing=0.06
)

# 3. Adiciona as curvas e áreas de cada ativo
for idx, ticker in enumerate(ativos_ordenados):
    row = (idx // 3) + 1
    col = (idx % 3) + 1
    
    setor = MAPA_SETORES_OFICIAL[ticker]
    cor_linha = paleta_oficial[setor]
    cor_fill = paleta_rgb_fill[setor]
    
    df_papel = df[df["ticker"] == ticker]
    
    fig_dd.add_trace(
        go.Scatter(
            x=df_papel["date"],
            y=df_papel["drawdown"],
            mode="lines",
            line=dict(color=cor_linha, width=1.5),
            fill="tozeroy",
            fillcolor=cor_fill,
            showlegend=False,
            hovertemplate="<b>%{x|%d/%m/%Y}</b><br>" +
                          f"Ativo: <b>{ticker}</b> ({setor})<br>" +
                          "Drawdown: <b>%{y:.1%}</b><extra></extra>"
        ),
        row=row, col=col
    )
    
    fig_dd.add_hline(y=0.0, line_dash="solid", line_color="#94A3B8", line_width=0.8, row=row, col=col)

# 4. Formatação de eixos idêntica para todos os painéis
for r in range(1, 5):
    for c in range(1, 4):
        fig_dd.update_yaxes(
            tickformat=".0%",
            range=[-0.65, 0.05],
            showgrid=True,
            gridcolor="rgba(226, 232, 240, 0.6)",
            row=r, col=c
        )
        fig_dd.update_xaxes(
            tickformat="%y",
            showgrid=True,
            gridcolor="rgba(226, 232, 240, 0.6)",
            row=r, col=c
        )

# 5. Marcadores para a legenda setorial na base
for setor, cor in paleta_oficial.items():
    fig_dd.add_trace(
        go.Scatter(
            x=[None], y=[None],
            mode="markers",
            name=setor,
            marker=dict(size=10, color=cor, symbol="square")
        )
    )

# 6. Layout com margem superior expandida (t=105) para afastar do subtítulo
fig_dd.update_layout(
    title=dict(
        text="<b>Curva Contínua de Drawdown Individual para os 12 Ativos da B3 (2021-2023)</b><br>"
             "<span style='font-size: 11px; color: #64748B;'>Painéis monocromáticos por departamento | Eixo Y padronizado para comparação direta de estresse de capital</span>",
        font=dict(size=14, color="#0F172A"),
        x=0.01,
        y=0.98
    ),
    template="plotly_white",
    height=890,
    hovermode="x unified",
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.04,
        xanchor="center",
        x=0.5,
        font=dict(size=10, color="#0F172A"),
        bgcolor="rgba(255, 255, 255, 0.9)",
        bordercolor="rgba(226, 232, 240, 0.8)",
        borderwidth=1
    ),
    margin=dict(r=25, t=105, b=60, l=45)  # t=105 garante respiro amplo
)

fig_dd.show()

# 7. Tabela oficial
print("\nRanking Oficial de Resiliência de Capital (Menor Drawdown Máximo enfrentado):")
df_ranking_dd = (
    df_stats[["Ticker", "Setor", "Max Drawdown (%)"]]
    .sort_values(by="Max Drawdown (%)", ascending=False)
    .reset_index(drop=True)
)
display(df_ranking_dd.round(2))


Ranking Oficial de Resiliência de Capital (Menor Drawdown Máximo enfrentado):


,Ticker,Setor,Max Drawdown (%)
0,EGIE3.SA,"Petróleo, Gás & Energia",-17.65
1,ABEV3.SA,"Financeiro, Consumo & Locação",-31.42
2,GGBR4.SA,Materiais Básicos & Siderurgia/Papel,-33.93
3,PRIO3.SA,"Petróleo, Gás & Energia",-34.32
4,ITUB4.SA,"Financeiro, Consumo & Locação",-35.87
5,RENT3.SA,"Financeiro, Consumo & Locação",-38.40
6,PETR4.SA,"Petróleo, Gás & Energia",-39.10
7,KLBN11.SA,Materiais Básicos & Siderurgia/Papel,-39.30
8,VALE3.SA,Materiais Básicos & Siderurgia/Papel,-41.34
9,BBDC4.SA,"Financeiro, Consumo & Locação",-46.05


### Pergunta de Negócio 5:
> *Como os preços de fechamento se comportaram frente às Médias Móveis Simples (SMA 21 de curto prazo e SMA 50 de médio prazo)?*

**Justificativa Analítica:**  
Inspeção de tendência e cruzamentos técnicos através de gráfico multilinha interativo sobrepondo a Cotação Ajustada (`adj_close`), a Média Móvel de Curto Prazo (`sma_21`) e a Média Móvel de Médio Prazo (`sma_50`) no ativo líder da carteira (PETR4.SA).

In [47]:
# Gráfico 5: Análise Multilinha de Médias Móveis Técnicas (PETR4.SA)
ticker_foco = "PETR4.SA"
df_papel = df[df["ticker"] == ticker_foco].sort_values("date").copy()

fig_sma = go.Figure()

# Cotação Fechamento Ajustado
fig_sma.add_trace(go.Scatter(
    x=df_papel["date"],
    y=df_papel["adj_close"],
    mode="lines",
    name="Preço Ajustado (R$)",
    line=dict(color="#1f77b4", width=2.2),
    hovertemplate="<b>%{x|%d/%m/%Y}</b><br>Preço: <b>R$ %{y:.2f}</b><extra></extra>"
))

# Média Móvel Simples 21 dias (Curto Prazo)
fig_sma.add_trace(go.Scatter(
    x=df_papel["date"],
    y=df_papel["sma_21"],
    mode="lines",
    name="SMA 21 (Curto Prazo)",
    line=dict(color="#ff7f0e", width=1.6, dash="dash"),
    hovertemplate="SMA 21: <b>R$ %{y:.2f}</b><extra></extra>"
))

# Média Móvel Simples 50 dias (Médio Prazo)
fig_sma.add_trace(go.Scatter(
    x=df_papel["date"],
    y=df_papel["sma_50"],
    mode="lines",
    name="SMA 50 (Médio Prazo)",
    line=dict(color="#2ca02c", width=1.6, dash="dot"),
    hovertemplate="SMA 50: <b>R$ %{y:.2f}</b><extra></extra>"
))

fig_sma.update_layout(
    title=dict(
        text=f"<b>Rastreamento de Tendência Técnica: Cotação Ajustada vs. Médias Móveis (SMA 21 e SMA 50) — {ticker_foco}</b><br>"
             f"<span style='font-size: 12px; color: #64748B;'>Cruzamentos de médias como suporte dinâmico e rastreadores de momento</span>",
        font=dict(size=15, color="#0F172A"),
        x=0.01
    ),
    xaxis=dict(title="Data do Pregão", showgrid=True, gridcolor="rgba(226, 232, 240, 0.7)", tickformat="%b/%Y"),
    yaxis=dict(title="Cotação em Reais (R$)", showgrid=True, gridcolor="rgba(226, 232, 240, 0.7)", tickprefix="R$ "),
    template="plotly_white",
    height=550,
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.18,
        xanchor="center",
        x=0.5,
        font=dict(size=10),
        bgcolor="rgba(255, 255, 255, 0.9)",
        bordercolor="rgba(226, 232, 240, 0.8)",
        borderwidth=1
    ),
    margin=dict(r=30, t=75, b=80, l=60)
)

fig_sma.show()

## 5. Exportação da Tabela Consolidada para a Documentação (README.md)
Geração em formato Markdown limpo para atualização direta da Seção 7 do arquivo `README.md`.

In [48]:
colunas_readme = ["Ticker", "Setor", "Retorno Acumulado (%)", "Volatilidade Anualizada (%)", "Índice Sharpe", "Max Drawdown (%)"]
tabela_readme = df_stats[colunas_readme].copy()

markdown_table = tabela_readme.to_markdown(index=False, floatfmt=".2f")
print("=== TABELA GERADA PARA A SEÇÃO 7 DO README.MD ===\n")
print(markdown_table)

=== TABELA GERADA PARA A SEÇÃO 7 DO README.MD ===

| Ticker    | Setor                                |   Retorno Acumulado (%) |   Volatilidade Anualizada (%) |   Índice Sharpe |   Max Drawdown (%) |
|:----------|:-------------------------------------|------------------------:|------------------------------:|----------------:|-------------------:|
| PETR4.SA  | Petróleo, Gás & Energia              |                  228.69 |                         39.42 |            5.80 |             -39.10 |
| PRIO3.SA  | Petróleo, Gás & Energia              |                  208.32 |                         46.78 |            4.45 |             -34.32 |
| EMBJ3.SA  | Bens de Capital & Indústria          |                  167.50 |                         46.82 |            3.58 |             -58.27 |
| GGBR4.SA  | Materiais Básicos & Siderurgia/Papel |                   30.79 |                         35.56 |            0.87 |             -33.93 |
| EGIE3.SA  | Petróleo, Gás & Energia            